In [ ]:
# %% [markdown]
# # 5. 텔롭 그루핑 검증 (ByteTrack + Gradio)

# %% 셀 1: imports & config
import os
import sys
import json
import types
import cv2
import numpy as np
import pandas as pd
import torch
import gradio as gr
from PIL import Image, ImageDraw, ImageFont
from collections import Counter, defaultdict

# ──────────────────────────────────────────────
# lap 호환 shim: lapx → scipy fallback
# ──────────────────────────────────────────────
try:
    import lapx
    sys.modules["lap"] = lapx
    print("✅ lapx 로드")
except ImportError:
    from scipy.optimize import linear_sum_assignment as _lsa
    _lap = types.ModuleType("lap")
    def _lapjv(cost, extend_cost=False, cost_limit=None):
        if cost.size == 0:
            return 0, np.empty(0, dtype=np.int64), np.empty(0, dtype=np.int64)
        if cost_limit is not None:
            mask = cost > cost_limit
            cost = cost.copy()
            cost[mask] = cost_limit + 1
        ri, ci = _lsa(cost)
        x = np.full(cost.shape[0], -1, dtype=np.int64)
        y = np.full(cost.shape[1], -1, dtype=np.int64)
        for r, c in zip(ri, ci):
            if cost_limit is None or cost[r, c] <= cost_limit:
                x[r] = c
                y[c] = r
        return sum(cost[r, c] for r, c in zip(ri, ci) if x[r] >= 0), x, y
    _lap.lapjv = _lapjv
    sys.modules["lap"] = _lap
    print("✅ scipy fallback shim 로드")

from bytetracker import BYTETracker

# ──────────────────────────────────────────────
# Config
# ──────────────────────────────────────────────
OCR_DIR = "./data/3_ocr_results"
TELOP_DIR = "./data/4_is_telop_results"
FRAME_DIR = "./data/2_frame_files"
FPS = 10

_DUMMY_DETS = torch.tensor([[0, 0, 1, 1, 0.001, 0]], dtype=torch.float32)

PALETTE = [
    (255, 56, 56), (255, 157, 56), (255, 235, 56), (56, 255, 56),
    (56, 255, 157), (56, 255, 235), (56, 157, 255), (56, 56, 255),
    (157, 56, 255), (235, 56, 255), (255, 56, 235), (255, 56, 157),
    (200, 100, 100), (100, 200, 100), (100, 100, 200), (200, 200, 100),
    (200, 100, 200), (100, 200, 200), (180, 180, 180), (128, 0, 0),
]

_FONT = None
def _get_font(size=18):
    global _FONT
    if _FONT is not None and _FONT.size == size:
        return _FONT
    for p in [
        "/usr/share/fonts/truetype/noto/NotoSansCJK-Regular.ttc",
        "/usr/share/fonts/truetype/noto/NotoSansCJKkr-Regular.otf",
        "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
        "/usr/share/fonts/truetype/nanum/NanumGothic.ttf",
    ]:
        if os.path.exists(p):
            _FONT = ImageFont.truetype(p, size)
            return _FONT
    _FONT = ImageFont.load_default()
    return _FONT

print("✅ imports 완료")


# %% 셀 2: 유틸리티 함수

def get_color(tid):
    return PALETTE[tid % len(PALETTE)]


def xywha_to_tlbr(cx, cy, w, h, angle):
    rad = np.radians(angle)
    ca, sa = abs(np.cos(rad)), abs(np.sin(rad))
    bw, bh = w * ca + h * sa, w * sa + h * ca
    return [cx - bw / 2, cy - bh / 2, cx + bw / 2, cy + bh / 2]


def xywha_to_corners(cx, cy, w, h, angle):
    rad = np.radians(angle)
    cos_a, sin_a = np.cos(rad), np.sin(rad)
    dx = np.array([w / 2 * cos_a, w / 2 * sin_a])
    dy = np.array([-h / 2 * sin_a, h / 2 * cos_a])
    return np.array([
        [cx - dx[0] - dy[0], cy - dx[1] - dy[1]],
        [cx + dx[0] - dy[0], cy + dx[1] - dy[1]],
        [cx + dx[0] + dy[0], cy + dx[1] + dy[1]],
        [cx - dx[0] + dy[0], cy - dx[1] + dy[1]],
    ], dtype=np.int32)


def iou_batch(a, b):
    a, b = np.asarray(a, dtype=np.float64), np.asarray(b, dtype=np.float64)
    if a.size == 0 or b.size == 0:
        return np.zeros((len(a), len(b)), dtype=np.float64)
    ix1 = np.maximum(a[:, None, 0], b[None, :, 0])
    iy1 = np.maximum(a[:, None, 1], b[None, :, 1])
    ix2 = np.minimum(a[:, None, 2], b[None, :, 2])
    iy2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.maximum(ix2 - ix1, 0) * np.maximum(iy2 - iy1, 0)
    aa = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1])
    ab = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / np.maximum(aa[:, None] + ab[None, :] - inter, 1e-6)


def list_channels():
    if not os.path.isdir(TELOP_DIR):
        return []
    return sorted(d for d in os.listdir(TELOP_DIR) if os.path.isdir(os.path.join(TELOP_DIR, d)))


def list_videos(channel):
    if not channel:
        return []
    ch = os.path.join(TELOP_DIR, channel)
    return sorted(f[:-8] for f in os.listdir(ch) if f.endswith(".parquet")) if os.path.isdir(ch) else []


print("✅ 유틸리티 로드 완료")


# %% 셀 3: 트래킹 실행 + 시각화

def run_tracking(channel, video_name, track_thresh, track_buffer, match_thresh):
    ocr_pq = os.path.join(OCR_DIR, channel, f"{video_name}.parquet")
    telop_pq = os.path.join(TELOP_DIR, channel, f"{video_name}.parquet")

    df_ocr = pd.read_parquet(ocr_pq).sort_values("frame_num").reset_index(drop=True)
    df_telop = pd.read_parquet(telop_pq).sort_values("frame_num").reset_index(drop=True)
    telop_map = {int(r["frame_num"]): r for _, r in df_telop.iterrows()}

    tracker = BYTETracker(
        track_thresh=track_thresh,
        track_buffer=int(track_buffer),
        match_thresh=match_thresh,
        frame_rate=FPS,
    )

    frame_results = {}
    track_info = defaultdict(lambda: {"frames": [], "texts": [], "xywhas": [], "confs": []})

    for _, ocr_row in df_ocr.iterrows():
        fnum = int(ocr_row["frame_num"])
        telop_row = telop_map.get(fnum)

        if telop_row is None:
            tracker.update(_DUMMY_DETS, None)
            frame_results[fnum] = []
            continue

        ocr_texts = json.loads(ocr_row["ocr_texts"]) if isinstance(ocr_row["ocr_texts"], str) else ocr_row["ocr_texts"]
        ocr_xywha = json.loads(ocr_row["ocr_xywha"]) if isinstance(ocr_row["ocr_xywha"], str) else ocr_row["ocr_xywha"]
        is_telop = json.loads(telop_row["is_telop"]) if isinstance(telop_row["is_telop"], str) else telop_row["is_telop"]
        telop_p = json.loads(telop_row["telop_p"]) if isinstance(telop_row["telop_p"], str) else telop_row["telop_p"]

        det_tlbrs, det_scores, det_texts, det_xywhas = [], [], [], []
        for idx in range(min(len(ocr_texts), len(is_telop))):
            if not is_telop[idx]:
                continue
            s = telop_p[idx] if telop_p[idx] is not None else 0.5
            bbox = ocr_xywha[idx]
            det_tlbrs.append(xywha_to_tlbr(*bbox))
            det_scores.append(s)
            det_texts.append(ocr_texts[idx])
            det_xywhas.append(bbox)

        if det_tlbrs:
            dets = torch.zeros((len(det_tlbrs), 6), dtype=torch.float32)
            dets[:, 0:4] = torch.tensor(det_tlbrs, dtype=torch.float32)
            dets[:, 4] = torch.tensor(det_scores, dtype=torch.float32)
            dets[:, 5] = 0
        else:
            dets = _DUMMY_DETS

        outputs = tracker.update(dets, None)

        fr = []
        if isinstance(outputs, np.ndarray) and outputs.ndim == 2 and outputs.shape[0] > 0 and det_tlbrs:
            out_tlbrs = outputs[:, 0:4]
            det_arr = np.array(det_tlbrs)
            ious = iou_batch(out_tlbrs, det_arr)

            for oi in range(len(outputs)):
                tid = int(outputs[oi, 4])
                sc = float(outputs[oi, 6])
                best = np.argmax(ious[oi])
                if ious[oi, best] > 0.3:
                    fr.append({
                        "track_id": tid,
                        "xywha": det_xywhas[best],
                        "text": det_texts[best],
                        "score": sc,
                    })
                    track_info[tid]["frames"].append(fnum)
                    track_info[tid]["texts"].append(det_texts[best])
                    track_info[tid]["xywhas"].append(det_xywhas[best])
                    track_info[tid]["confs"].append(sc)

        frame_results[fnum] = fr

    segments = []
    for tid, info in sorted(track_info.items()):
        if not info["frames"]:
            continue
        frames = sorted(info["frames"])
        tc = Counter(info["texts"])
        segments.append({
            "track_id": tid,
            "start_frame": frames[0],
            "end_frame": frames[-1],
            "start_time": f"{frames[0] / FPS:.1f}s",
            "end_time": f"{frames[-1] / FPS:.1f}s",
            "duration": f"{(frames[-1] - frames[0]) / FPS:.1f}s",
            "text": tc.most_common(1)[0][0],
            "texts_all": list(tc.keys()),
            "n_frames": len(frames),
            "avg_conf": round(np.mean(info["confs"]), 3),
        })

    frame_nums = sorted(df_ocr["frame_num"].tolist())
    return frame_results, segments, frame_nums


def draw_frame(channel, video_name, frame_num, frame_results):
    img_path = os.path.join(FRAME_DIR, channel, video_name, f"frame_{frame_num:08d}.jpg")
    if os.path.exists(img_path):
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    else:
        img = np.zeros((720, 1280, 3), dtype=np.uint8)

    for r in frame_results.get(frame_num, []):
        color = get_color(r["track_id"])
        corners = xywha_to_corners(*r["xywha"])
        cv2.polylines(img, [corners], True, color, 3)

    pil_img = Image.fromarray(img)
    draw = ImageDraw.Draw(pil_img)
    font = _get_font(18)

    for r in frame_results.get(frame_num, []):
        tid = r["track_id"]
        color = get_color(tid)
        corners = xywha_to_corners(*r["xywha"])

        label = f"T{tid} | {r['text'][:20]} | {r['score']:.2f}"
        bbox = draw.textbbox((0, 0), label, font=font)
        tw, th = bbox[2] - bbox[0], bbox[3] - bbox[1]

        lx = max(0, int(corners[:, 0].min()))
        ly = max(0, int(corners[:, 1].min()) - th - 8)

        draw.rectangle([lx - 2, ly - 2, lx + tw + 4, ly + th + 4], fill=color)
        draw.text((lx, ly), label, fill=(255, 255, 255), font=font)

    n = len(frame_results.get(frame_num, []))
    info = f"Frame {frame_num}  |  {frame_num / FPS:.1f}s  |  {n} telop(s)"
    info_bbox = draw.textbbox((0, 0), info, font=font)
    draw.rectangle([0, 0, info_bbox[2] + 20, info_bbox[3] + 12], fill=(0, 0, 0))
    draw.text((10, 4), info, fill=(255, 255, 255), font=font)

    return np.array(pil_img)


print("✅ 트래킹 + 시각화 함수 로드 완료")


# %% 셀 4: Gradio App

_state = {"fr": {}, "segs": [], "fnums": [], "ch": None, "vn": None}


def on_channel_change(channel):
    vs = list_videos(channel)
    return gr.update(choices=vs, value=vs[0] if vs else None)


def on_run(channel, video_name, thresh, buf, match):
    if not channel or not video_name:
        return None, "⚠️ 채널과 영상을 선택하세요.", gr.update(maximum=0, value=0), ""

    fr, segs, fnums = run_tracking(channel, video_name, thresh, int(buf), match)
    _state.update({"fr": fr, "segs": segs, "fnums": fnums, "ch": channel, "vn": video_name})

    lines = [
        f"{'TID':>4}  {'시작':>7}  {'종료':>7}  {'길이':>6}  {'프레임':>4}  {'conf':>5}  텍스트",
        "─" * 80,
    ]
    for s in segs:
        lines.append(
            f"  T{s['track_id']:<3d}  {s['start_time']:>7}  {s['end_time']:>7}"
            f"  {s['duration']:>6}  {s['n_frames']:>4d}  {s['avg_conf']:.3f}"
            f"  {s['text'][:40]}"
        )
    seg_text = f"✅ {len(segs)}개 텔롭 세그먼트  |  {len(fnums)}개 프레임\n\n" + "\n".join(lines)

    max_idx = max(len(fnums) - 1, 0)
    img = draw_frame(channel, video_name, fnums[0], fr) if fnums else None
    detail = _frame_detail(0)

    return img, seg_text, gr.update(maximum=max_idx, value=0), detail


def _frame_detail(idx):
    if not _state["fnums"]:
        return ""
    idx = min(int(idx), len(_state["fnums"]) - 1)
    fnum = _state["fnums"][idx]
    results = _state["fr"].get(fnum, [])
    lines = [f"Frame {fnum}  |  {fnum / FPS:.1f}s  |  {len(results)}개 텔롭"]
    for r in results:
        lines.append(f"  T{r['track_id']}  \"{r['text'][:30]}\"  conf={r['score']:.3f}")
    return "\n".join(lines)


def on_slider(val):
    if not _state["fnums"]:
        return None, ""
    idx = min(int(val), len(_state["fnums"]) - 1)
    fnum = _state["fnums"][idx]
    img = draw_frame(_state["ch"], _state["vn"], fnum, _state["fr"])
    return img, _frame_detail(idx)


channels = list_channels()

with gr.Blocks(title="텔롭 그루핑 검증 (ByteTrack)", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🎬 텔롭 그루핑 검증 (ByteTrack)")
    gr.Markdown(
        "채널/영상 선택 → **트래킹 실행** → 프레임 슬라이더로 결과 확인.\n"
        "각 텔롭은 track_id 별 색상으로 표시되며, 하단에 시간 구간별 그루핑 결과가 나옵니다."
    )

    with gr.Row():
        with gr.Column(scale=1):
            dd_ch = gr.Dropdown(choices=channels, label="채널",
                                value=channels[0] if channels else None)
            dd_vn = gr.Dropdown(choices=list_videos(channels[0]) if channels else [],
                                label="영상")

            gr.Markdown("### ⚙️ ByteTrack 파라미터")
            sl_thresh = gr.Slider(0.1, 0.9, value=0.5, step=0.05,
                                  label="track_thresh (텔롭 확률 임계)")
            sl_buf = gr.Slider(1, 50, value=1, step=1,
                               label="track_buffer (미검출 허용 프레임, 10=1초)")
            sl_match = gr.Slider(0.3, 1.0, value=0.7, step=0.05,
                                 label="match_thresh (IoU 매칭 임계)")

            btn = gr.Button("🚀 트래킹 실행", variant="primary", size="lg")

        with gr.Column(scale=3):
            img_out = gr.Image(label="프레임 미리보기", height=520)
            sl_frame = gr.Slider(0, 100, value=0, step=1, label="프레임 탐색")
            txt_detail = gr.Textbox(label="현재 프레임 상세", lines=4, interactive=False)

    txt_seg = gr.Textbox(
        label="📋 텔롭 세그먼트 (ByteTrack 그루핑 결과)",
        lines=18, interactive=False,
    )

    dd_ch.change(on_channel_change, [dd_ch], [dd_vn])
    btn.click(on_run, [dd_ch, dd_vn, sl_thresh, sl_buf, sl_match],
              [img_out, txt_seg, sl_frame, txt_detail])
    sl_frame.change(on_slider, [sl_frame], [img_out, txt_detail])

demo.launch(server_name="0.0.0.0", server_port=7860, share=False)

In [1]:
"""
5. 텔롭 그루핑 — 배치 실행 (멀티프로세스)
  Input:  3_ocr_results + 4_is_telop_results
  Output: 5_telop_segments/{channel}/{video_name}.parquet (frame_num, track_id)
"""

import os
import sys
import json
import glob
import types
import numpy as np
import pandas as pd
import torch
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm.auto import tqdm

# ──────────────────────────────────────────────
# lap 호환 shim
# ──────────────────────────────────────────────
try:
    import lapx
    sys.modules["lap"] = lapx
except ImportError:
    from scipy.optimize import linear_sum_assignment as _lsa

    def _lapjv(cost, extend_cost=False, cost_limit=None):
        if cost.size == 0:
            return 0, np.empty(0, dtype=np.int64), np.empty(0, dtype=np.int64)
        if cost_limit is not None:
            mask = cost > cost_limit
            cost = cost.copy()
            cost[mask] = cost_limit + 1
        ri, ci = _lsa(cost)
        x = np.full(cost.shape[0], -1, dtype=np.int64)
        y = np.full(cost.shape[1], -1, dtype=np.int64)
        for r, c in zip(ri, ci):
            if cost_limit is None or cost[r, c] <= cost_limit:
                x[r] = c
                y[c] = r
        return sum(cost[r, c] for r, c in zip(ri, ci) if x[r] >= 0), x, y

    _lap = types.ModuleType("lap")
    _lap.__path__ = []
    _lap.lapjv = _lapjv

    _lap_sub = types.ModuleType("lap._lapjv_wp")
    _lap_sub.lapjv = _lapjv
    _lap._lapjv_wp = _lap_sub

    sys.modules["lap"] = _lap
    sys.modules["lap._lapjv_wp"] = _lap_sub

from bytetracker import BYTETracker
from bytetracker.basetrack import BaseTrack

print("✅ imports 완료")

# ──────────────────────────────────────────────
# Config
# ──────────────────────────────────────────────
OCR_DIR = "./data/3_ocr_results"
TELOP_DIR = "./data/4_is_telop_results"
OUTPUT_DIR = "./data/5_telop_segments"
FPS = 10
NUM_WORKERS = 32

TRACK_THRESH = 0.5
TRACK_BUFFER = 1
MATCH_THRESH = 0.7

_DUMMY_DETS = torch.tensor([[0, 0, 1, 1, 0.001, 0]], dtype=torch.float32)


# ──────────────────────────────────────────────
# 유틸리티
# ──────────────────────────────────────────────
def xywha_to_tlbr(cx, cy, w, h, angle):
    rad = np.radians(angle)
    ca, sa = abs(np.cos(rad)), abs(np.sin(rad))
    bw, bh = w * ca + h * sa, w * sa + h * ca
    return [cx - bw / 2, cy - bh / 2, cx + bw / 2, cy + bh / 2]


def iou_batch(a, b):
    a, b = np.asarray(a, dtype=np.float64), np.asarray(b, dtype=np.float64)
    if a.size == 0 or b.size == 0:
        return np.zeros((len(a), len(b)), dtype=np.float64)
    ix1 = np.maximum(a[:, None, 0], b[None, :, 0])
    iy1 = np.maximum(a[:, None, 1], b[None, :, 1])
    ix2 = np.minimum(a[:, None, 2], b[None, :, 2])
    iy2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.maximum(ix2 - ix1, 0) * np.maximum(iy2 - iy1, 0)
    aa = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1])
    ab = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / np.maximum(aa[:, None] + ab[None, :] - inter, 1e-6)


# ──────────────────────────────────────────────
# 단일 영상 처리 (워커에서 실행)
# ──────────────────────────────────────────────
def process_one_video(ocr_pq, telop_pq):
    BaseTrack._count = 0

    df_ocr = pd.read_parquet(ocr_pq).sort_values("frame_num").reset_index(drop=True)
    df_telop = pd.read_parquet(telop_pq).sort_values("frame_num").reset_index(drop=True)
    telop_map = {int(r["frame_num"]): r for _, r in df_telop.iterrows()}

    tracker = BYTETracker(
        track_thresh=TRACK_THRESH,
        track_buffer=TRACK_BUFFER,
        match_thresh=MATCH_THRESH,
        frame_rate=FPS,
    )
    tracker.det_thresh = tracker.track_thresh

    rows = []

    for _, ocr_row in df_ocr.iterrows():
        fnum = int(ocr_row["frame_num"])

        ocr_texts = json.loads(ocr_row["ocr_texts"]) if isinstance(ocr_row["ocr_texts"], str) else ocr_row["ocr_texts"]
        ocr_xywha = json.loads(ocr_row["ocr_xywha"]) if isinstance(ocr_row["ocr_xywha"], str) else ocr_row["ocr_xywha"]
        n_regions = len(ocr_texts)

        telop_row = telop_map.get(fnum)
        if telop_row is None:
            tracker.update(_DUMMY_DETS, None)
            rows.append((fnum, json.dumps([-1] * n_regions)))
            continue

        is_telop = json.loads(telop_row["is_telop"]) if isinstance(telop_row["is_telop"], str) else telop_row["is_telop"]
        telop_p = json.loads(telop_row["telop_p"]) if isinstance(telop_row["telop_p"], str) else telop_row["telop_p"]

        det_tlbrs, det_scores, det_indices = [], [], []
        for idx in range(min(n_regions, len(is_telop))):
            if not is_telop[idx]:
                continue
            s = telop_p[idx] if telop_p[idx] is not None else 0.5
            bbox = ocr_xywha[idx]
            det_tlbrs.append(xywha_to_tlbr(*bbox))
            det_scores.append(s)
            det_indices.append(idx)

        if det_tlbrs:
            dets = torch.zeros((len(det_tlbrs), 6), dtype=torch.float32)
            dets[:, 0:4] = torch.tensor(det_tlbrs, dtype=torch.float32)
            dets[:, 4] = torch.tensor(det_scores, dtype=torch.float32)
            dets[:, 5] = 0
        else:
            dets = _DUMMY_DETS

        outputs = tracker.update(dets, None)

        track_ids = [-1] * n_regions

        if isinstance(outputs, np.ndarray) and outputs.ndim == 2 and outputs.shape[0] > 0 and det_tlbrs:
            out_tlbrs = outputs[:, 0:4]
            det_arr = np.array(det_tlbrs)
            ious = iou_batch(out_tlbrs, det_arr)

            for oi in range(len(outputs)):
                tid = int(outputs[oi, 4])
                best = np.argmax(ious[oi])
                if ious[oi, best] > 0.3:
                    orig_idx = det_indices[best]
                    track_ids[orig_idx] = tid

        rows.append((fnum, json.dumps(track_ids)))

    # is_telop=True & track_id=-1 → 새 track_id 부여
    max_tid = 0
    for _, tid_json in rows:
        for t in json.loads(tid_json):
            max_tid = max(max_tid, t)

    new_rows = []
    for fnum, tid_json in rows:
        tids = json.loads(tid_json)
        telop_row = telop_map.get(fnum)
        is_telop = json.loads(telop_row["is_telop"]) if telop_row is not None and isinstance(telop_row["is_telop"], str) else (telop_row["is_telop"] if telop_row is not None else [])

        for idx in range(len(tids)):
            if tids[idx] == -1 and idx < len(is_telop) and is_telop[idx]:
                max_tid += 1
                tids[idx] = max_tid

        new_rows.append((fnum, json.dumps(tids)))

    return new_rows

# ──────────────────────────────────────────────
# 배치 실행
# ──────────────────────────────────────────────
os.makedirs(OUTPUT_DIR, exist_ok=True)

stats = {"done": 0, "errors": 0, "skip": 0}

channels = sorted([d for d in os.listdir(TELOP_DIR) if os.path.isdir(os.path.join(TELOP_DIR, d))])

for channel in tqdm(channels, desc="📁 채널"):
    ch_telop = os.path.join(TELOP_DIR, channel)
    ch_ocr = os.path.join(OCR_DIR, channel)
    ch_out = os.path.join(OUTPUT_DIR, channel)
    os.makedirs(ch_out, exist_ok=True)

    parquets = sorted([
        os.path.join(ch_telop, f) 
        for f in os.listdir(ch_telop) 
        if f.endswith(".parquet")
    ])


    ch_tasks = []
    for telop_pq in parquets:
        fname = os.path.basename(telop_pq)
        ocr_pq = os.path.join(ch_ocr, fname)
        out_pq = os.path.join(ch_out, fname)
        if not os.path.exists(ocr_pq):
            continue
        if os.path.exists(out_pq):
            stats["skip"] += 1
            continue
        ch_tasks.append((ocr_pq, telop_pq, out_pq, fname[:-8]))

    if not ch_tasks:
        continue

    with ProcessPoolExecutor(max_workers=NUM_WORKERS) as pool:
        futures = {
            pool.submit(process_one_video, ocr, tel): (out, vn)
            for ocr, tel, out, vn in ch_tasks
        }
        for fut in tqdm(as_completed(futures), total=len(futures),
                        desc=f"  🎬 {channel[:20]}", leave=False):
            out_pq, video_name = futures[fut]
            try:
                rows = fut.result()
                pd.DataFrame(rows, columns=["frame_num", "track_id"]).to_parquet(out_pq, index=False)
                stats["done"] += 1
            except Exception as e:
                stats["errors"] += 1
                tqdm.write(f"  ❌ {channel}/{video_name}: {e}")

print(f"\n✅ 완료: {stats['done']:,}개  |  스킵: {stats['skip']:,}개  |  오류: {stats['errors']}개")

/home/taeyoung/miniconda3/envs/chi/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ imports 완료


📁 채널: 100%|██████████| 664/664 [16:36<00:00,  1.50s/it]


✅ 완료: 66,400개  |  스킵: 0개  |  오류: 0개


In [2]:
# ──────────────────────────────────────────────
# 검증 (32코어 병렬)
# ──────────────────────────────────────────────
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm.auto import tqdm

NUM_VERIFY_WORKERS = 32


def verify_one(pq_path, channel, fname):
    df = pd.read_parquet(pq_path)
    n_frames = len(df)
    n_tracked = 0
    n_untracked = 0
    n_telop_but_untracked = 0

    rel = os.path.join(channel, fname)
    telop_pq = os.path.join(TELOP_DIR, rel)
    mismatch = None

    # telop 결과 로드
    telop_map = {}
    if os.path.exists(telop_pq):
        df_telop = pd.read_parquet(telop_pq)
        if n_frames != len(df_telop):
            mismatch = (rel, n_frames, len(df_telop))
        telop_map = {int(r["frame_num"]): r for _, r in df_telop.iterrows()}

    for _, row in df.iterrows():
        fnum = int(row["frame_num"])
        tids = json.loads(row["track_id"])
        n_tracked += sum(1 for t in tids if t >= 0)
        n_untracked += sum(1 for t in tids if t < 0)

        telop_row = telop_map.get(fnum)
        if telop_row is not None:
            is_telop = json.loads(telop_row["is_telop"]) if isinstance(telop_row["is_telop"], str) else telop_row["is_telop"]
            for idx in range(min(len(tids), len(is_telop))):
                if is_telop[idx] and tids[idx] < 0:
                    n_telop_but_untracked += 1

    return n_frames, n_tracked, n_untracked, n_telop_but_untracked, mismatch


tasks = []
for channel in sorted(os.listdir(OUTPUT_DIR)):
    ch_out = os.path.join(OUTPUT_DIR, channel)
    if not os.path.isdir(ch_out):
        continue
    for fname in sorted(f for f in os.listdir(ch_out) if f.endswith(".parquet")):
        tasks.append((os.path.join(ch_out, fname), channel, fname))

total_files = 0
total_frames = 0
total_tracked = 0
total_untracked = 0
mismatches = []

total_telop_but_untracked = 0

with ProcessPoolExecutor(max_workers=NUM_VERIFY_WORKERS) as pool:
    futures = {pool.submit(verify_one, p, c, f): (c, f) for p, c, f in tasks}
    for fut in tqdm(as_completed(futures), total=len(futures), desc="🔍 검증"):
        n_frames, n_tracked, n_untracked, n_tbu, mismatch = fut.result()
        total_files += 1
        total_frames += n_frames
        total_tracked += n_tracked
        total_untracked += n_untracked
        total_telop_but_untracked += n_tbu
        if mismatch:
            mismatches.append(mismatch)

print(f"\n📊 검증 결과:")
print(f"   파일 수: {total_files:,}")
print(f"   프레임 수: {total_frames:,}")
print(f"   tracked bbox: {total_tracked:,}")
print(f"   untracked bbox: {total_untracked:,}")
print(f"   ⚠️ is_telop=True & track_id=-1: {total_telop_but_untracked:,}")

if mismatches:
    print(f"   ⚠️ 행 수 불일치: {len(mismatches)}건")
    for m in mismatches[:5]:
        print(f"      {m[0]}: output={m[1]} vs telop={m[2]}")
else:
    print(f"   ✅ 행 수 불일치 없음")

🔍 검증: 100%|██████████| 66400/66400 [00:46<00:00, 1434.15it/s]



📊 검증 결과:
   파일 수: 66,400
   프레임 수: 27,525,206
   tracked bbox: 92,875,783
   untracked bbox: 40,720,610
   ⚠️ is_telop=True & track_id=-1: 0
   ✅ 행 수 불일치 없음


In [3]:
"""
5. 텔롭 그루핑 검증 — is_telop=True & track_id=-1 케이스 확인
"""

import os
import sys
import json
import cv2
import numpy as np
import pandas as pd
import gradio as gr
from PIL import Image, ImageDraw, ImageFont

# ──────────────────────────────────────────────
# Config
# ──────────────────────────────────────────────
OCR_DIR = "./data/3_ocr_results"
TELOP_DIR = "./data/4_is_telop_results"
SEGMENT_DIR = "./data/5_telop_segments"
FRAME_DIR = "./data/2_frame_files"
FPS = 10

COLOR_TRACKED = (56, 255, 56)      # 초록: tracked
COLOR_UNTRACKED = (255, 56, 56)    # 빨강: is_telop=True & track_id=-1
COLOR_SCENE = (128, 128, 128)      # 회색: is_telop=False

_FONT = None
def _get_font(size=18):
    global _FONT
    if _FONT is not None and _FONT.size == size:
        return _FONT
    for p in [
        "/usr/share/fonts/truetype/noto/NotoSansCJK-Regular.ttc",
        "/usr/share/fonts/truetype/noto/NotoSansCJKkr-Regular.otf",
        "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
        "/usr/share/fonts/truetype/nanum/NanumGothic.ttf",
    ]:
        if os.path.exists(p):
            _FONT = ImageFont.truetype(p, size)
            return _FONT
    _FONT = ImageFont.load_default()
    return _FONT


def xywha_to_corners(cx, cy, w, h, angle):
    rad = np.radians(angle)
    cos_a, sin_a = np.cos(rad), np.sin(rad)
    dx = np.array([w / 2 * cos_a, w / 2 * sin_a])
    dy = np.array([-h / 2 * sin_a, h / 2 * cos_a])
    return np.array([
        [cx - dx[0] - dy[0], cy - dx[1] - dy[1]],
        [cx + dx[0] - dy[0], cy + dx[1] - dy[1]],
        [cx + dx[0] + dy[0], cy + dx[1] + dy[1]],
        [cx - dx[0] + dy[0], cy - dx[1] + dy[1]],
    ], dtype=np.int32)


# ──────────────────────────────────────────────
# 채널/영상 목록
# ──────────────────────────────────────────────
def list_channels():
    if not os.path.isdir(SEGMENT_DIR):
        return []
    return sorted(d for d in os.listdir(SEGMENT_DIR) if os.path.isdir(os.path.join(SEGMENT_DIR, d)))


def list_videos(channel):
    if not channel:
        return []
    ch = os.path.join(SEGMENT_DIR, channel)
    if not os.path.isdir(ch):
        return []
    return sorted(f[:-8] for f in os.listdir(ch) if f.endswith(".parquet"))


# ──────────────────────────────────────────────
# 데이터 로드 + 문제 프레임 필터링
# ──────────────────────────────────────────────
def load_video_data(channel, video_name):
    """
    3개 parquet 조인 후, is_telop=True & track_id=-1인 프레임만 필터.
    Returns:
        all_frames: 전체 프레임 데이터 (dict[fnum] → {...})
        problem_fnums: is_telop=True & track_id=-1 가 있는 프레임 번호 리스트
        stats: 통계 dict
    """
    fname = f"{video_name}.parquet"
    ocr_pq = os.path.join(OCR_DIR, channel, fname)
    telop_pq = os.path.join(TELOP_DIR, channel, fname)
    seg_pq = os.path.join(SEGMENT_DIR, channel, fname)

    df_ocr = pd.read_parquet(ocr_pq).sort_values("frame_num").reset_index(drop=True)
    df_telop = pd.read_parquet(telop_pq).sort_values("frame_num").reset_index(drop=True)
    df_seg = pd.read_parquet(seg_pq).sort_values("frame_num").reset_index(drop=True)

    telop_map = {int(r["frame_num"]): r for _, r in df_telop.iterrows()}
    seg_map = {int(r["frame_num"]): r for _, r in df_seg.iterrows()}

    all_frames = {}
    problem_fnums = []
    total_telop = 0
    total_tracked = 0
    total_untracked = 0

    for _, ocr_row in df_ocr.iterrows():
        fnum = int(ocr_row["frame_num"])
        ocr_texts = json.loads(ocr_row["ocr_texts"]) if isinstance(ocr_row["ocr_texts"], str) else ocr_row["ocr_texts"]
        ocr_xywha = json.loads(ocr_row["ocr_xywha"]) if isinstance(ocr_row["ocr_xywha"], str) else ocr_row["ocr_xywha"]

        telop_row = telop_map.get(fnum)
        seg_row = seg_map.get(fnum)

        is_telop = json.loads(telop_row["is_telop"]) if telop_row is not None and isinstance(telop_row["is_telop"], str) else (telop_row["is_telop"] if telop_row is not None else [])
        track_ids = json.loads(seg_row["track_id"]) if seg_row is not None and isinstance(seg_row["track_id"], str) else (seg_row["track_id"] if seg_row is not None else [])

        regions = []
        has_problem = False
        for idx in range(len(ocr_texts)):
            it = is_telop[idx] if idx < len(is_telop) else False
            tid = track_ids[idx] if idx < len(track_ids) else -1

            if it:
                total_telop += 1
                if tid >= 0:
                    total_tracked += 1
                else:
                    total_untracked += 1
                    has_problem = True

            regions.append({
                "text": ocr_texts[idx],
                "xywha": ocr_xywha[idx],
                "is_telop": it,
                "track_id": tid,
            })

        all_frames[fnum] = regions
        if has_problem:
            problem_fnums.append(fnum)

    stats = {
        "total_frames": len(df_ocr),
        "total_telop": total_telop,
        "total_tracked": total_tracked,
        "total_untracked": total_untracked,
        "problem_frames": len(problem_fnums),
    }

    return all_frames, sorted(problem_fnums), stats


# ──────────────────────────────────────────────
# 프레임 시각화
# ──────────────────────────────────────────────
def draw_frame(channel, video_name, frame_num, regions):
    img_path = os.path.join(FRAME_DIR, channel, video_name, f"frame_{frame_num:08d}.jpg")
    if os.path.exists(img_path):
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    else:
        img = np.zeros((720, 1280, 3), dtype=np.uint8)

    # bbox 그리기
    for r in regions:
        if r["is_telop"] and r["track_id"] < 0:
            color = COLOR_UNTRACKED
            thickness = 3
        elif r["is_telop"] and r["track_id"] >= 0:
            color = COLOR_TRACKED
            thickness = 2
        else:
            color = COLOR_SCENE
            thickness = 1

        corners = xywha_to_corners(*r["xywha"])
        cv2.polylines(img, [corners], True, color, thickness)

    # 텍스트 라벨 (PIL)
    pil_img = Image.fromarray(img)
    draw = ImageDraw.Draw(pil_img)
    font = _get_font(16)

    for r in regions:
        if r["is_telop"] and r["track_id"] < 0:
            color = COLOR_UNTRACKED
            label = f"❌ {r['text'][:20]}"
        elif r["is_telop"] and r["track_id"] >= 0:
            color = COLOR_TRACKED
            label = f"T{r['track_id']} | {r['text'][:20]}"
        else:
            continue  # scene 텍스트는 라벨 생략

        corners = xywha_to_corners(*r["xywha"])
        bbox = draw.textbbox((0, 0), label, font=font)
        tw, th = bbox[2] - bbox[0], bbox[3] - bbox[1]
        lx = max(0, int(corners[:, 0].min()))
        ly = max(0, int(corners[:, 1].min()) - th - 6)
        draw.rectangle([lx - 2, ly - 2, lx + tw + 4, ly + th + 4], fill=color)
        draw.text((lx, ly), label, fill=(255, 255, 255), font=font)

    # 상단 바
    n_prob = sum(1 for r in regions if r["is_telop"] and r["track_id"] < 0)
    n_ok = sum(1 for r in regions if r["is_telop"] and r["track_id"] >= 0)
    info = f"Frame {frame_num} | {frame_num / FPS:.1f}s | ❌ untracked: {n_prob}  ✅ tracked: {n_ok}"
    info_bbox = draw.textbbox((0, 0), info, font=font)
    draw.rectangle([0, 0, info_bbox[2] + 20, info_bbox[3] + 12], fill=(0, 0, 0))
    draw.text((10, 4), info, fill=(255, 255, 255), font=font)

    return np.array(pil_img)


# ──────────────────────────────────────────────
# Gradio App
# ──────────────────────────────────────────────
_state = {"frames": {}, "problem_fnums": [], "ch": None, "vn": None}


def on_channel_change(channel):
    vs = list_videos(channel)
    return gr.update(choices=vs, value=vs[0] if vs else None)


def on_load(channel, video_name):
    if not channel or not video_name:
        return None, "⚠️ 채널과 영상을 선택하세요.", gr.update(maximum=0, value=0), ""

    all_frames, problem_fnums, stats = load_video_data(channel, video_name)
    _state.update({"frames": all_frames, "problem_fnums": problem_fnums, "ch": channel, "vn": video_name})

    summary = (
        f"📊 전체 telop bbox: {stats['total_telop']:,}\n"
        f"   ✅ tracked: {stats['total_tracked']:,}\n"
        f"   ❌ untracked (is_telop=True & track_id=-1): {stats['total_untracked']:,}\n"
        f"   문제 프레임: {stats['problem_frames']:,} / {stats['total_frames']:,}"
    )

    if not problem_fnums:
        return None, summary + "\n\n✅ untracked 텔롭 없음!", gr.update(maximum=0, value=0), ""

    max_idx = len(problem_fnums) - 1
    img = draw_frame(channel, video_name, problem_fnums[0], all_frames[problem_fnums[0]])
    detail = _frame_detail(0)

    return img, summary, gr.update(maximum=max_idx, value=0), detail


def _frame_detail(idx):
    if not _state["problem_fnums"]:
        return ""
    idx = min(int(idx), len(_state["problem_fnums"]) - 1)
    fnum = _state["problem_fnums"][idx]
    regions = _state["frames"].get(fnum, [])
    lines = [f"Frame {fnum} | {fnum / FPS:.1f}s | 문제 프레임 {idx + 1}/{len(_state['problem_fnums'])}"]
    for r in regions:
        if r["is_telop"] and r["track_id"] < 0:
            lines.append(f"  ❌ \"{r['text'][:30]}\"")
        elif r["is_telop"] and r["track_id"] >= 0:
            lines.append(f"  ✅ T{r['track_id']} \"{r['text'][:30]}\"")
    return "\n".join(lines)


def on_slider(val):
    if not _state["problem_fnums"]:
        return None, ""
    idx = min(int(val), len(_state["problem_fnums"]) - 1)
    fnum = _state["problem_fnums"][idx]
    img = draw_frame(_state["ch"], _state["vn"], fnum, _state["frames"][fnum])
    return img, _frame_detail(idx)


channels = list_channels()

with gr.Blocks(title="텔롭 untracked 검증", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🔍 is_telop=True & track_id=-1 검증")
    gr.Markdown(
        "빨간색 = untracked (is_telop=True인데 track_id=-1)\n"
        "초록색 = tracked (정상)\n"
        "회색 = scene 텍스트 (is_telop=False)"
    )

    with gr.Row():
        with gr.Column(scale=1):
            dd_ch = gr.Dropdown(choices=channels, label="채널",
                                value=channels[0] if channels else None)
            dd_vn = gr.Dropdown(choices=list_videos(channels[0]) if channels else [],
                                label="영상")
            btn = gr.Button("🔍 로드", variant="primary", size="lg")
            txt_stats = gr.Textbox(label="📊 통계", lines=5, interactive=False)

        with gr.Column(scale=3):
            img_out = gr.Image(label="프레임", height=520)
            sl_frame = gr.Slider(0, 100, value=0, step=1, label="문제 프레임 탐색")
            txt_detail = gr.Textbox(label="프레임 상세", lines=5, interactive=False)

    dd_ch.change(on_channel_change, [dd_ch], [dd_vn])
    btn.click(on_load, [dd_ch, dd_vn], [img_out, txt_stats, sl_frame, txt_detail])
    sl_frame.change(on_slider, [sl_frame], [img_out, txt_detail])

demo.launch(server_name="0.0.0.0", server_port=7860, share=False)

/tmp/ipykernel_337152/1991917736.py:273: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title="텔롭 untracked 검증", theme=gr.themes.Soft()) as demo:


* Running on local URL:  http://0.0.0.0:7860
* To create a public link, set `share=True` in `launch()`.
